# Yaya-125M — Kaggle Training

**Before running:** Settings → Accelerator → **GPU T4 x2** (or P100) → Save

## Session flow
| Session | What happens |
|---------|-------------|
| 1st | Tokenize Wikipedia (~15 min) + pretrain 5000 steps (~10 hrs) |
| 2nd+ | Download checkpoint dataset → auto-resume pretrain or start SFT |

## Checkpoint persistence between sessions
Kaggle doesn't have persistent storage like Drive. Between sessions:
1. After each session ends, go to **Output tab** → download `yaya-checkpoints.zip`
2. Upload it as a **Kaggle Dataset** (once)
3. On the next session, attach that dataset → checkpoint is auto-detected

Or skip persistence and just train start-to-finish in one ~12h session (T4 x2 is faster).


In [ ]:
# Step 1: Check GPU
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT FOUND')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
if torch.cuda.device_count() > 1:
    print(f'  ({torch.cuda.device_count()} GPUs available)')

In [ ]:
# Step 2: Setup
import os
!git clone https://github.com/jaylink-coder/miss-yaya.git /kaggle/working/miss-yaya 2>/dev/null || \
    (cd /kaggle/working/miss-yaya && git stash && git pull origin main)
!pip install -q -r /kaggle/working/miss-yaya/yaya-ai/requirements-kaggle.txt
os.chdir('/kaggle/working/miss-yaya/yaya-ai')
print('Working dir:', os.getcwd())

In [ ]:
# Step 3: Pretrain Yaya-1B
# - First run: tokenizes all Wikipedia (~60 min), then trains
# - Re-runs: skips tokenization, auto-resumes from latest checkpoint
# - Checkpoints: /kaggle/working/yaya-checkpoints-1b/
!python scripts/kaggle_1b.py

In [ ]:
# Step 4: SFT — run after pretraining converges
# Downloads OpenHermes 100K + fine-tunes on combined SFT data
# Checkpoints: /kaggle/working/yaya-sft-1b-checkpoints/
!python scripts/kaggle_sft_1b.py

In [ ]:
# Step 5: Evaluate
import glob
sft_ckpts = sorted(glob.glob('/kaggle/working/yaya-sft-1b-checkpoints/checkpoint-*'))
best = sft_ckpts[-1] if sft_ckpts else None
if best:
    import os
    os.system(
        f'python scripts/eval_yaya.py '
        f'--model_config configs/model/yaya_1b.yaml '
        f'--checkpoint {best} --chat'
    )
else:
    print('No SFT checkpoint found.')